In [1]:
import torch
from transformers import (
    T5Tokenizer, 
    T5ForConditionalGeneration, 
    Seq2SeqTrainingArguments, 
    Seq2SeqTrainer
)
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset

T5-smallのモデルを作ってみる

In [2]:
model_name = "t5-small"
print(f"Loading {model_name}...")
tokenizer = T5Tokenizer.from_pretrained(model_name, legacy=False)
model = T5ForConditionalGeneration.from_pretrained(model_name)

Loading t5-small...


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

ここがmainのsetting

In [3]:
lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM, # T5のようなEncoder-Decoderモデルを指定
    r=8,                             # 論文第7章: 極小のランクで十分 (トンネルの太さ)
    lora_alpha=16,                   # スケーリングファクター用の定数 (マスターボリューム)
    target_modules=["q", "v"],       # 論文第7章: W_qとW_vにバイパスを付けるのが最強らしいので
    lora_dropout=0.05,               # 過学習防止のドロップアウト
)

In [4]:
model = get_peft_model(model, lora_config)

モデルに関して、全パラメータは、6000万個あるらしいが、訓練するのはLoRAの300000個弱のパラメータのみ

In [5]:
model.print_trainable_parameters()

trainable params: 294,912 || all params: 60,801,536 || trainable%: 0.4850


前回のT5Modelとの比較をしてみるので、条件は同じで行く

In [6]:
from datasets import load_dataset
dataset = load_dataset("cnn_dailymail", "3.0.0", split="train[:1000]")
dataset = dataset.train_test_split(test_size=0.1)


Using the latest cached version of the dataset since cnn_dailymail couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration '3.0.0' at /home/popo/.cache/huggingface/datasets/cnn_dailymail/3.0.0/0.0.0/96df5e686bee6baa90b8bee7c28b81fa3fa6223d (last modified on Thu Jun 25 23:35:38 2026).


In [7]:
prefix = "summarize: "
max_input_length = 512
max_target_length = 128

def preprocess_function(examples):
    # 1. 入力文章の先頭にプレフィックスを付ける
    inputs = [prefix + doc for doc in examples["article"]]
    
    # 2. エンコーダ向けの入力をトークナイズ
    model_inputs = tokenizer(
        inputs, 
        max_length=max_input_length, 
        truncation=True
    )

    labels = tokenizer(
        text_target=examples["highlights"], 
        max_length=max_target_length, 
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# データセット全体に前処理を一括適用 (バッチ処理で高速化)
tokenized_datasets = dataset.map(preprocess_function, batched=True)

Map:   0%|          | 0/900 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [8]:
tokenized_datasets

DatasetDict({
    train: Dataset({
        features: ['article', 'highlights', 'id', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 900
    })
    test: Dataset({
        features: ['article', 'highlights', 'id', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 100
    })
})

In [20]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./t5-small-cnn-dailymail", # モデルの保存先
    eval_strategy="epoch",             # 評価のタイミング（1エポック終わるごとに評価）
    learning_rate=2e-5,                    # 学習率（T5ファインチューニングの標準的な値）
    per_device_train_batch_size=8,         # 訓練時のバッチサイズ
    per_device_eval_batch_size=8,          # 評価時のバッチサイズ
    weight_decay=0.01,                     # 過学習を防ぐための重み減衰
    save_total_limit=3,                    # 保存するチェックポイントの最大数
    num_train_epochs=3,                    # データセットを何周学習するか
    predict_with_generate=True,            # 【必須】評価時に実際にテキストを生成(デコード)させる
    fp16=False,                            # GPU(CUDA)環境があり、Tensorコアが使える場合はTrueにすると高速化・省メモリ
)

In [10]:
from transformers import DataCollatorForSeq2Seq


data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

In [21]:
import evaluate
import numpy as np
rouge = evaluate.load("rouge")
def compute_metrics(eval_pred):
    predictions, labels = eval_pred#valデータに対する、回答、ラベルが返ってくる
    
    # モデルの予測IDをテキスト（文字列）にデコード
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    
    # 【重要】labelsの中にある「-100（Loss無視フラグ）」を、pad_token_idに戻す
    # （-100のままだとトークナイザがデコードできずにエラーになるため）
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # ROUGEスコアを計算
    result = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
    
    # 読みやすいように数値を100倍してパーセント表示にする
    return {k: round(v * 100, 4) for k, v in result.items()}

In [22]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],   # 先ほど分割したtestデータを検証(Validation)として使用
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

精度は若干低くなる。
このLoRA自体、パラメータのメモリ削減をしないといけない（自作モデル）時に、役立つものである。
これも一つの手法として、覚えておくのが大事

In [23]:
trainer.train()

Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum
1,No log,1.877995,23.075200,9.230400,18.332700,18.322700
2,No log,1.878070,23.037300,9.233100,18.335700,18.325500
3,No log,1.877983,23.075200,9.230400,18.332700,18.322700


TrainOutput(global_step=339, training_loss=2.0458824135209257, metrics={'train_runtime': 42.0415, 'train_samples_per_second': 64.222, 'train_steps_per_second': 8.063, 'total_flos': 367868982067200.0, 'train_loss': 2.0458824135209257, 'epoch': 3.0})